# Q3. A partir de que semestre eu consigo cursar cada optativa do meu curso?

**Pergunta.** Em cada semestre do curso, quantas e quais optativas o aluno já pode
cursar, e qual é a sequência mínima de disciplinas até cada uma?

**Dados.** Quatro tabelas da coleta do JúpiterWeb (31/10/2026): `grade_curricular`,
`requisitos`, `disciplinas` e `oferecimentos`. Entram os oito cursos e habilitações do
conjunto de dados. O Núcleo Geral da Matemática fica de fora porque suas obrigatórias
já estão nas grades do bacharelado e da licenciatura.

**Como medimos.** Consideramos o aluno que segue a grade ideal e não reprova:

- uma **obrigatória** é concluída no seu semestre ideal;
- uma **optativa** é liberada no semestre seguinte ao da conclusão dos seus requisitos.
  Sem requisito no curso, ela está liberada desde o 1º semestre;
- valem o `requisito` e o `requisito fraco`, e a `indicação de conjunto` não precisa vir
  antes, porque é cursada junto;
- quando há grupos alternativos de requisitos, usamos o que libera a optativa mais cedo;
- se a optativa exige outra optativa, esta é cursada no primeiro semestre possível.

A **sequência mínima** reúne todas as disciplinas que precisam vir antes, direta ou
indiretamente. Cada optativa fica em uma destas situações: sem requisito, liberada só
com obrigatórias, exige outra optativa antes, ou depende de disciplina fora da grade.


In [1]:
import sys
if "google.colab" in sys.modules:
    import importlib.util, os, subprocess
    REPO = "https://github.com/vmoreli/viscomp-projeto"
    PASTA = "/content/viscomp-projeto"
    if not os.path.exists(PASTA):
        subprocess.run(["git", "clone", "--depth", "1", REPO, PASTA], check=True)
    os.chdir(os.path.join(PASTA, "notebooks"))
    faltando = [m for m in ("plotly",) if importlib.util.find_spec(m) is None]
    if faltando:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)


In [2]:
from pathlib import Path

import pandas as pd

DADOS = Path("../dados/processado")
if not DADOS.exists():
    DADOS = Path("dados/processado")

grade = pd.read_csv(DADOS / "grade_curricular.csv", dtype={"sigla": str})
req = pd.read_csv(DADOS / "requisitos.csv", dtype={"requisito": str, "disciplina": str})
cursos = pd.read_csv(DADOS / "cursos.csv")
disc = pd.read_csv(DADOS / "disciplinas.csv", usecols=["sigla", "nome"], dtype={"sigla": str})
ofer = pd.read_csv(DADOS / "oferecimentos.csv", dtype={"sigla": str})
NOME = disc.set_index("sigla")["nome"].to_dict()
COM_TURMA = set(ofer["sigla"])

req["codhab"] = req["codhab"].fillna(0).astype(int)
cursos = cursos[cursos["codhab"] != 100].reset_index(drop=True)
cursos["rotulo"] = (cursos["nome_curso"].str.replace("Bacharelado em ", "Bach. ")
                    .str.replace("Licenciatura em ", "Lic. "))
TIPO = {"optativa_eletiva": "eletiva", "optativa_livre": "livre"}


def analisar_curso(codcur, codhab):
    g = grade[(grade.codcur == codcur) & (grade.codhab == codhab)]
    ideal = dict(zip(g["sigla"], g["semestre_ideal"].astype(int)))
    obrig = set(g.loc[g["tipo_vinculo"] == "obrigatoria", "sigla"])
    r = req[(req.codcur == codcur) & (req.codhab == codhab)]

    grupos = {}
    for d, sub in r.groupby("disciplina"):
        gs = [list(zip(x.loc[x["tipo_requisito"] != "indicação de conjunto", "requisito"],
                       x.loc[x["tipo_requisito"] != "indicação de conjunto", "tipo_requisito"]))
              for _, x in sub.groupby("grupo_alternativa")]
        if all(gs):
            grupos[d] = gs

    memo = {}

    def melhor_grupo(d, visitando):
        melhor = None
        for gr in grupos.get(d, []):
            if not all(p in ideal for p, _ in gr):
                continue
            tempos = [semestre_conclusao(p, visitando) for p, _ in gr]
            if any(t is None for t in tempos):
                continue
            chave = (max(tempos) + 1, len(gr))
            if melhor is None or chave < melhor[0]:
                melhor = (chave, gr)
        return melhor

    def semestre_conclusao(p, visitando):
        return ideal[p] if p in obrig else liberacao(p, visitando)

    def liberacao(d, visitando=frozenset()):
        if d in memo:
            return memo[d]
        if d in visitando:
            return None
        if d not in grupos:
            memo[d] = 1
            return 1
        m = melhor_grupo(d, visitando | {d})
        memo[d] = m[0][0] if m else None
        return memo[d]

    def sequencia(d):
        vistas, pilha = {}, [d]
        while pilha:
            x = pilha.pop()
            m = melhor_grupo(x, frozenset({x})) if x in grupos else None
            for p, tipo in (m[1] if m else []):
                if p not in vistas:
                    vistas[p] = None
                    pilha.append(p)
        return sorted(vistas, key=lambda p: (semestre_conclusao(p, frozenset()) or 99, p))

    linhas = []
    for o in g[g["tipo_vinculo"] != "obrigatoria"].itertuples():
        d = o.sigla
        t = liberacao(d)
        alternativas = grupos.get(d, [])
        so_obrig = any(all(p in obrig for p, _ in gr) for gr in alternativas)
        lib_ob = min((max(ideal[p] for p, _ in gr) + 1 for gr in alternativas
                      if all(p in obrig for p, _ in gr)), default=None)
        if d not in grupos:
            lib_ob = 1
            situacao = "livre"
        elif t is None:
            situacao = "fora"
        elif so_obrig:
            situacao = "obrigatorias"
        else:
            situacao = "optativa"
        seq = sequencia(d) if t is not None and d in grupos else []
        fora = sorted({p for gr in alternativas for p, _ in gr if p not in ideal})
        m = melhor_grupo(d, frozenset({d})) if d in grupos else None
        linhas.append({
            "s": d, "nome": NOME.get(d, d), "tipo": TIPO[o.tipo_vinculo], "ideal": int(o.semestre_ideal),
            "lib": t, "lib_ob": lib_ob, "sit": situacao, "turma": d in COM_TURMA, "alts": len(alternativas),
            "dir": [[p, tp] for p, tp in (m[1] if m else [])],
            "seq": [[p, NOME.get(p, p), "ob" if p in obrig else "op", semestre_conclusao(p, frozenset())] for p in seq],
            "fora": [[p, NOME.get(p, p)] for p in fora],
        })
    return linhas, max(ideal.values())


DADOS_VIS = {"ordem": [], "cursos": {}}
for c in cursos.itertuples():
    k = f"{c.codcur}-{c.codhab}"
    linhas, ult = analisar_curso(c.codcur, c.codhab)
    DADOS_VIS["ordem"].append(k)
    DADOS_VIS["cursos"][k] = dict(nome=c.rotulo, ultimo=ult, opts=linhas)
DADOS_VIS["ordem"].sort(key=lambda k: (k != "55041-0", DADOS_VIS["cursos"][k]["nome"]))


def resumo(k):
    o = pd.DataFrame(DADOS_VIS["cursos"][k]["opts"])
    meio = (DADOS_VIS["cursos"][k]["ultimo"] + 1) // 2
    alcancaveis = o[o["lib"].notna()]
    return {
        "curso": DADOS_VIS["cursos"][k]["nome"], "optativas": len(o),
        "1º sem.": int((o["lib"] == 1).sum()), "até o 3º": int((o["lib"] <= 3).sum()),
        "até o 5º": int((o["lib"] <= 5).sum()), "% até o 5º": round(100 * (o["lib"] <= 5).mean()),
        "só com outra optativa": int((o["sit"] == "optativa").sum()),
        "fora da grade": int((o["sit"] == "fora").sum()), "com turma 2026/2": int(o["turma"].sum()),
        "ideal antes de liberar": int((alcancaveis["ideal"] < alcancaveis["lib"]).sum()),
        "antecedência mediana": (alcancaveis["ideal"] - alcancaveis["lib"]).median(),
        "seq. mediana": alcancaveis["seq"].map(len).median(), "seq. máxima": alcancaveis["seq"].map(len).max(),
    }


pd.DataFrame([resumo(k) for k in DADOS_VIS["ordem"]])


,curso,optativas,1º sem.,até o 3º,até o 5º,% até o 5º,só com outra optativa,fora da grade,com turma 2026/2,ideal antes de liberar,antecedência mediana,seq. mediana,seq. máxima
0,Bach. Ciências de Computação,95,29,45,70,74,7,1,43,0,3.0,2.0,8
1,Bach. Ciência de Dados,78,16,37,67,86,18,0,32,0,1.0,3.0,7
2,Bach. Estatística e Ciência de Dados,45,13,21,42,93,2,0,15,0,3.0,3.0,6
3,Bach. Matemática,61,23,46,60,98,19,0,20,0,3.0,1.0,8
4,Bach. Matemática Aplicada e Computação Científica,56,14,23,40,71,6,0,26,0,1.0,2.5,6
5,Bach. Sistemas de Informação,40,15,21,33,82,4,0,21,0,4.0,2.0,6
6,Engenharia de Computação,145,59,74,96,66,15,0,63,0,3.0,1.0,12
7,Lic. Matemática,51,16,29,40,78,8,1,15,2,2.0,1.0,6


## Visualizações coordenadas

1. **Quantas optativas ficam disponíveis?** **Curva acumulada em degraus**: o semestre é
   ordenado, e a contagem só cresce, porque uma optativa liberada continua liberada. A
   curva mostra quando as opções se abrem. A linha azul conta o que as obrigatórias
   liberam, e a pontilhada inclui as optativas que exigem outra optativa antes.
2. **Quais são essas optativas?** **Lista filtrável**: o aluno procura nomes, e não
   compara quantidades, então uma lista atende melhor que um gráfico. As recém-liberadas
   vêm primeiro.
3. **O que preciso cursar antes?** **Tabela da sequência mínima**, em ordem de semestre,
   com os requisitos diretos marcados.

**Interação**, seguindo *overview first, zoom and filter, then details on demand*:

- **menu de curso**, **controle deslizante de semestre**, **filtro de turma em 2026/2** e
  **busca por nome ou sigla**: consultas dinâmicas;
- **clique em um semestre da curva** → o controle e a lista acompanham;
- **clique em uma optativa** → a tabela mostra a sequência mínima, e um losango marca na
  curva o semestre em que ela é liberada;
- **dicas ao passar o mouse**: total acumulado e optativas liberadas naquele semestre.

**Acessibilidade.** As duas curvas se distinguem também pelo traço (contínuo e
pontilhado), o semestre escolhido é marcado por um círculo e por uma linha vertical, e a
lista e a tabela trazem os mesmos dados em texto.


In [3]:
JS_PAINEL = r"""
const COR = {tinta: "#0b0b0b", tinta2: "#52514e", muda: "#898781", grade: "#e1e0d9",
             azul: "#2a78d6", laranja: "#c2570c", fundo: "#fcfcfb"};
const st = {curso: D.ordem[0], n: 3, turma: false, aba: "sim", busca: "", sel: null};
const $ = (s) => root.querySelector(s);
const fig = $(".fig"), selCurso = $(".curso"), slider = $(".sem"), busca = $(".busca");
const cfg = {displayModeBar: false, responsive: true};
const fonte = {family: "Inter, Arial, sans-serif", size: 12, color: COR.tinta2};
const esc = (s) => String(s).replace(/[&<>"]/g, (c) => ({"&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;"}[c]));
const sem = (n) => `${n}º`;
const simples = (s) => String(s).normalize("NFD").replace(/[̀-ͯ]/g, "").toLowerCase();
const curso = () => D.cursos[st.curso];
const universo = () => curso().opts.filter((o) => !st.turma || o.turma);
const liberada = (o) => o.lib !== null && o.lib <= st.n;

// ---------- controles (consultas dinâmicas) ----------
D.ordem.forEach((k) => selCurso.add(new Option(D.cursos[k].nome, k)));
selCurso.addEventListener("change", () => {
  st.curso = selCurso.value;
  st.n = Math.min(st.n, curso().ultimo);
  st.sel = null;
  tudo();
});
slider.addEventListener("input", () => { st.n = Number(slider.value); tudo(); });
$(".turma").addEventListener("change", (e) => { st.turma = e.target.checked; st.sel = null; tudo(); });
busca.addEventListener("input", () => { st.busca = busca.value; desenhaLista(); });
root.querySelectorAll(".abas button").forEach((b) => b.addEventListener("click", () => {
  st.aba = b.dataset.v;
  desenhaLista();
}));

// ---------- visão 1: curva acumulada ----------
function curvas() {
  const ult = curso().ultimo, xs = Array.from({length: ult}, (_, i) => i + 1), u = universo();
  return {xs, u,
    ob: xs.map((t) => u.filter((o) => o.lib_ob !== null && o.lib_ob <= t).length),
    tudo: xs.map((t) => u.filter((o) => o.lib !== null && o.lib <= t).length)};
}
function desenhaCurva() {
  const {xs, u, ob, tudo: td} = curvas();
  const total = u.length, ult = xs.length;
  const dica = xs.map((t, i) => `<b>${sem(t)} semestre</b><br>${td[i]} de ${total} optativas podem ser cursadas` +
    `<br>${ob[i]} só com obrigatórias<br>${td[i] - (i ? td[i - 1] : 0)} liberadas neste semestre<br><i>clique para ir a este semestre</i>`);
  const tracos = [
    {x: xs, y: td, customdata: dica, type: "scatter", mode: "lines+markers", line: {shape: "hv", color: COR.laranja, width: 2, dash: "dot"},
     marker: {size: 8, color: COR.laranja, line: {color: COR.fundo, width: 2}}, hovertemplate: "%{customdata}<extra></extra>"},
    {x: xs, y: ob, customdata: dica, type: "scatter", mode: "lines+markers", line: {shape: "hv", color: COR.azul, width: 2.5},
     marker: {size: 8, color: COR.azul, line: {color: COR.fundo, width: 2}}, hovertemplate: "%{customdata}<extra></extra>"},
    {x: [st.n], y: [td[st.n - 1]], type: "scatter", mode: "markers+text", hoverinfo: "skip", cliponaxis: false,
     marker: {size: 13, color: COR.fundo, line: {color: COR.tinta, width: 2.5}},
     text: [`<b>${td[st.n - 1]}</b>`], textposition: st.n >= ult - 1 ? "bottom center" : "top left", textfont: {size: 13, color: COR.tinta}}];
  const o = st.sel && curso().opts.find((x) => x.s === st.sel);
  if (o && o.lib !== null && o.lib <= ult) {
    tracos.push({x: [o.lib], y: [td[o.lib - 1]], type: "scatter", mode: "markers", hoverinfo: "skip", cliponaxis: false,
      marker: {size: 12, symbol: "diamond", color: COR.tinta, line: {color: COR.fundo, width: 2}}});
  }
  const lay = {
    height: 400, margin: {l: 50, r: 22, t: 16, b: 48}, font: fonte, showlegend: false,
    paper_bgcolor: COR.fundo, plot_bgcolor: COR.fundo, hovermode: "closest",
    xaxis: {title: {text: "semestre em que o aluno está"}, tickvals: xs, ticktext: xs.map(sem), range: [0.6, ult + 0.4],
            showgrid: false, zeroline: false, fixedrange: true},
    yaxis: {title: {text: "optativas que pode cursar"}, range: [0, Math.max(total, 1) * 1.08], gridcolor: COR.grade, zeroline: false, fixedrange: true},
    shapes: [
      {type: "line", xref: "paper", x0: 0, x1: 1, y0: total, y1: total, line: {color: COR.muda, width: 1, dash: "dash"}},
      {type: "line", x0: st.n, x1: st.n, yref: "paper", y0: 0, y1: 1, line: {color: COR.tinta, width: 1}}],
    annotations: [{xref: "paper", x: 1, xanchor: "right", y: total, yanchor: "bottom", showarrow: false,
                   text: `total de optativas: ${total}`, font: {size: 11, color: COR.tinta2}}]};
  return Plotly.react(fig, tracos, lay, cfg);
}

// ---------- resumo ----------
function desenhaResumo() {
  const {u, tudo: td} = curvas(), total = u.length, v = td[st.n - 1];
  const faltam = u.filter((o) => o.sit === "optativa").length;
  const fora = u.filter((o) => o.sit === "fora").length;
  $(".resumo").innerHTML = `<b>${esc(curso().nome)}, ${sem(st.n)} semestre:</b> ` +
    (total ? `você já pode cursar <b>${v} de ${total}</b> optativas (${Math.round(100 * v / total)}%)` : "nenhuma optativa neste filtro") +
    (st.turma ? " com turma em 2026/2" : "") +
    "." + (faltam ? ` ${faltam} ${faltam === 1 ? "exige" : "exigem"} outra optativa antes.` : "") +
    (fora ? ` ${fora} ${fora === 1 ? "depende" : "dependem"} de disciplina fora da grade.` : "");
  $(".rot-sem").textContent = `${sem(st.n)} semestre`;
}

// ---------- visão 2: lista de optativas ----------
function linha(o) {
  const nova = o.lib === st.n && st.n > 1;
  const quando = liberada(o) ? "" : (o.lib === null ? "sem caminho na grade" : `a partir do ${sem(o.lib)}`);
  return `<li><button class="op${o.s === st.sel ? " marcada" : ""}" data-s="${o.s}" aria-pressed="${o.s === st.sel}">
    <span><b>${o.s}</b> ${esc(o.nome)}${nova ? ' <span class="tag">nova</span>' : ""}</span>
    <small>${o.tipo}${o.turma ? " · turma em 2026/2" : ""}${quando ? " · " + quando : ""}</small></button></li>`;
}
function desenhaLista() {
  const t = simples(st.busca.trim());
  const combina = (o) => !t || simples(o.s + " " + o.nome).includes(t);
  const u = universo();
  const sim = u.filter(liberada), nao = u.filter((o) => !liberada(o));
  root.querySelectorAll(".abas button").forEach((b) => {
    b.classList.toggle("on", st.aba === b.dataset.v);
    b.textContent = b.dataset.v === "sim" ? `Já posso cursar (${sim.length})` : `Ainda não (${nao.length})`;
  });
  // A busca procura nas duas abas: o aluno quer achar a optativa, liberada ou não.
  const ordena = (a, b) => (liberada(b) - liberada(a)) || (b.lib === st.n) - (a.lib === st.n) ||
    (liberada(a) ? 0 : (a.lib ?? 99) - (b.lib ?? 99)) || a.s.localeCompare(b.s);
  const lista = (t ? u : st.aba === "sim" ? sim : nao).filter(combina).sort(ordena);
  $(".abas").classList.toggle("buscando", !!t);
  $(".lista").innerHTML = lista.length ? lista.map(linha).join("") : '<li class="muda">Nenhuma optativa encontrada.</li>';
}

// ---------- visão 3: sequência mínima ----------
function desenhaDetalhe() {
  const el = $(".detalhe"), o = st.sel && curso().opts.find((x) => x.s === st.sel);
  if (!o) {
    el.innerHTML = '<p class="muda">Clique em uma optativa da lista para ver o que é preciso cursar antes dela.</p>';
    return;
  }
  const direto = Object.fromEntries(o.dir);
  let corpo;
  if (o.sit === "livre") {
    corpo = "<p>Sem requisito neste curso: pode ser cursada desde o 1º semestre.</p>";
  } else if (o.sit === "fora") {
    corpo = `<p>Todas as alternativas de requisito incluem disciplina fora da grade deste curso:</p><ul class="fora">` +
      o.fora.map(([s, n]) => `<li><b>${s}</b> ${esc(n)}</li>`).join("") + "</ul>";
  } else {
    corpo = `<p>Pode ser cursada a partir do <b>${sem(o.lib)} semestre</b>, depois de <b>${o.seq.length}</b> disciplina(s):</p>` +
      `<table><thead><tr><th>Semestre</th><th>Disciplina</th><th>Vínculo</th></tr></thead><tbody>` +
      o.seq.map(([s, n, t, q]) => `<tr><td>${q ? sem(q) : "–"}</td><td><b>${s}</b> ${esc(n)}` +
        (direto[s] ? ` <span class="tag">${direto[s]} direto</span>` : "") + `</td><td>${t === "ob" ? "obrigatória" : "optativa"}</td></tr>`).join("") +
      "</tbody></table>" +
      (o.alts > 1 ? `<p class="muda">Há ${o.alts} alternativas de requisito; esta é a que libera mais cedo.</p>` : "");
  }
  el.innerHTML = `<h4>${o.s} · ${esc(o.nome)}</h4><p class="muda">Optativa ${o.tipo} · semestre ideal na grade: ${sem(o.ideal)}` +
    `${o.turma ? " · com turma em 2026/2" : " · sem turma em 2026/2"}</p>` + corpo;
}

function tudo() {
  selCurso.value = st.curso;
  slider.max = curso().ultimo;
  slider.value = st.n;
  desenhaResumo();
  desenhaLista();
  desenhaDetalhe();
  return desenhaCurva();
}

// ---------- ligação entre as visões ----------
$(".lista").addEventListener("click", (e) => {
  const b = e.target.closest("button.op");
  if (!b) return;
  st.sel = st.sel === b.dataset.s ? null : b.dataset.s;
  desenhaLista();
  desenhaDetalhe();
  desenhaCurva();
});
tudo().then(() => {
  fig.on("plotly_click", (ev) => {
    const p = ev.points.find((x) => x.curveNumber < 2);
    if (p) { st.n = p.x; tudo(); }
  });
});
"""

CSS_PAINEL = """
  #__ID__ { font-family: Inter, Arial, sans-serif; color: #52514e; background: #fcfcfb; padding: 12px 8px; }
  #__ID__ .ctl { display: flex; flex-wrap: wrap; gap: 10px 22px; align-items: center; font-size: 14px; }
  #__ID__ select, #__ID__ .busca { font: inherit; padding: 4px 6px; border: 1px solid #c3c2b7; border-radius: 4px; background: white; color: #0b0b0b; max-width: 100%; }
  #__ID__ .sem { width: 170px; vertical-align: middle; accent-color: #2a78d6; }
  #__ID__ .rot-sem { font-weight: 600; color: #0b0b0b; display: inline-block; min-width: 88px; }
  #__ID__ .resumo { margin: 12px 0 4px; font-size: 15px; color: #0b0b0b; }
  #__ID__ .paineis { display: flex; flex-wrap: wrap; gap: 18px; align-items: flex-start; }
  #__ID__ .p1 { flex: 1.3 1 520px; min-width: 300px; }
  #__ID__ .p2 { flex: 1 1 360px; min-width: 290px; border-left: 1px solid #e1e0d9; padding-left: 16px; }
  #__ID__ h4 { margin: 8px 0 2px; font-size: 14px; color: #0b0b0b; font-weight: 600; }
  #__ID__ h4 small { font-weight: 400; color: #898781; }
  #__ID__ p { margin: 4px 0; font-size: 13px; color: #0b0b0b; }
  #__ID__ .muda { color: #898781; font-size: 12px; }
  #__ID__ .leg { display: flex; flex-wrap: wrap; gap: 4px 14px; font-size: 12px; margin-top: 2px; }
  #__ID__ .leg i { display: inline-block; vertical-align: middle; margin-right: 5px; }
  #__ID__ .abas { display: inline-flex; border: 1px solid #c3c2b7; border-radius: 4px; overflow: hidden; margin-bottom: 6px; }
  #__ID__ .abas button { font: inherit; font-size: 13px; padding: 4px 10px; border: 0; background: white; color: #52514e; cursor: pointer; }
  #__ID__ .abas button + button { border-left: 1px solid #c3c2b7; }
  #__ID__ .abas button.on { background: #2a78d6; color: white; }
  #__ID__ .abas.buscando { opacity: 0.45; }
  #__ID__ .busca { width: 100%; box-sizing: border-box; margin-bottom: 6px; font-size: 13px; }
  #__ID__ .lista { list-style: none; padding: 0; margin: 0; max-height: 400px; overflow-y: auto; border-top: 1px solid #e1e0d9; }
  #__ID__ .op { display: block; width: 100%; text-align: left; font: inherit; font-size: 13px; color: #0b0b0b; padding: 6px 4px; background: none; border: 0; border-bottom: 1px solid #e1e0d9; cursor: pointer; }
  #__ID__ .op:hover { background: #efeee8; }
  #__ID__ .op.marcada { background: #efeee8; box-shadow: inset 3px 0 0 #0b0b0b; }
  #__ID__ .op small { display: block; color: #898781; font-size: 11px; }
  #__ID__ .tag { font-size: 10px; font-weight: 600; color: #52514e; border: 1px solid #898781; border-radius: 3px; padding: 0 3px; white-space: nowrap; }
  #__ID__ .detalhe { margin-top: 10px; }
  #__ID__ table { width: 100%; border-collapse: collapse; font-size: 12px; color: #0b0b0b; }
  #__ID__ th { text-align: left; font-weight: 600; color: #52514e; padding: 3px 4px; border-bottom: 1px solid #c3c2b7; }
  #__ID__ td { padding: 3px 4px; border-bottom: 1px solid #e1e0d9; vertical-align: top; }
  #__ID__ .fora { margin: 4px 0; padding-left: 18px; font-size: 13px; color: #0b0b0b; }
"""

CORPO_PAINEL = """
<div class="ctl">
  <label>Curso <select class="curso"></select></label>
  <label>Estou no <span class="rot-sem"></span> <input class="sem" type="range" min="1" step="1" aria-label="Semestre em que o aluno está"></label>
  <label><input class="turma" type="checkbox"> Só optativas com turma em 2026/2</label>
</div>
<div class="resumo"></div>
<div class="paineis">
  <div class="p1"><h4>Quantas optativas ficam disponíveis a cada semestre? <small>clique em um semestre</small></h4>
    <div class="leg"><span><i style="width:18px;border-top:3px solid #2a78d6"></i>só com obrigatórias</span>
      <span><i style="width:18px;border-top:3px dotted #c2570c"></i>cursando outra optativa antes, se preciso</span>
      <span><i style="width:10px;height:10px;border-radius:50%;border:2px solid #0b0b0b"></i>semestre escolhido</span>
      <span><i style="width:9px;height:9px;transform:rotate(45deg);margin:0 7px 0 2px;background:#0b0b0b"></i>liberação da optativa selecionada</span></div>
    <div class="fig"></div>
    <div class="detalhe"></div></div>
  <div class="p2"><h4>Quais são essas optativas?</h4>
    <div class="abas"><button data-v="sim"></button><button data-v="nao"></button></div>
    <input class="busca" type="search" placeholder="Buscar por nome ou sigla" aria-label="Buscar optativa">
    <ul class="lista"></ul></div>
</div>
"""

import json, uuid
from IPython.display import HTML, display

EMBUTIR_PLOTLY = False

if EMBUTIR_PLOTLY:
    from plotly.offline import get_plotlyjs
    tag_plotly = f"<script>{get_plotlyjs()}</script>"
else:
    tag_plotly = '<script src="https://cdn.jsdelivr.net/npm/plotly.js-dist-min@2.35.2/plotly.min.js"></script>'

ID = "q3_" + uuid.uuid4().hex[:8]
HTML_PAINEL = (
    f'<div id="{ID}" class="q3">\n<style>' + CSS_PAINEL.replace("__ID__", ID) + "</style>" + CORPO_PAINEL + "</div>\n"
    + tag_plotly + "\n<script>\n(function() {\n"
    + f"const D = {json.dumps(DADOS_VIS, ensure_ascii=False)};\n"
    + f'const root = document.getElementById("{ID}");\n'
    + JS_PAINEL + "\n})();\n</script>\n")

with open("q3_interativo.html", "w", encoding="utf-8") as f:
    f.write("<!doctype html><meta charset='utf-8'><title>Q3</title>" + HTML_PAINEL)
display(HTML(HTML_PAINEL))


## Leitura

*Para reproduzir cada ponto, escolha o curso e o semestre indicados e clique na optativa.*

**Resposta curta.** A maior parte das optativas fica disponível antes do semestre indicado na grade: até o 5º semestre, o aluno já pode cursar de 66 % a 98 % delas, conforme o
curso. Da metade do curso em diante, os requisitos já não limitam a maioria das escolhas, mas em alguns
cursos é preciso planejar uma optativa antes da outra.

**1. As opções se abrem cedo.** No 1º semestre, de 21 % a 41 % das optativas já não têm
requisito. Até o 5º, são 66 % em Engenharia de Computação (96 de 145), 74 % em Ciências de
Computação (70 de 95), 82 % em Sistemas de Informação (33 de 40), 86 % em Ciência de Dados
(67 de 78) e 98 % no bacharelado em Matemática (60 de 61).

**2. A grade indica as optativas mais tarde do que os requisitos exigem.** Em cada curso, a
optativa típica é liberada de 1 a 4 semestres antes do semestre ideal da grade: 4 em Sistemas
de Informação e 3 em Ciências de Computação. O semestre ideal é uma sugestão, e o aluno que
se interessa por uma área pode antecipá-la.

**3. Em alguns cursos, uma optativa abre caminho para outra.** No bacharelado em Matemática,
19 das 61 optativas (31 %) não são liberadas só pelas obrigatórias, e em Ciência de Dados são
18 das 78 (23 %). A distância entre as duas curvas mostra esse efeito. Em Ciências de
Computação são 7 de 95, e em Estatística e Ciência de Dados, 2 de 45.

**4. As sequências costumam ser curtas, com exceções.** A sequência mínima mediana tem de 1 a
3 disciplinas. Em Engenharia de Computação, porém, optativas como SEL0461 (Fundamentos de
Mecânica Quântica para Engenharia 2) exigem 12 disciplinas antes e só são liberadas no 7º
semestre. Quem quer essa área precisa começar a planejar cedo.

**5. Há casos que merecem revisão.** Em Ciências de Computação, SSC0159 só pode ser cursada com
SSC0143, que não tem registro na tabela `disciplinas`. Na licenciatura em Matemática, SMA0193 exige
SMA0305 (Álgebra I), que não está na grade do curso. Também nessa licenciatura, SMA0343 aparece
no 2º semestre ideal, mas pelos requisitos só é liberada no 5º.

**Em 2026/2.** Com o filtro de turma, só de 29 % (licenciatura em Matemática) a 53 % (Sistemas
de Informação) das optativas têm turma. Estar liberada não garante que a disciplina seja
oferecida naquele semestre.

**Limitações.**

- O modelo supõe um aluno que segue a grade ideal e não reprova. Reprovar pode atrasar a
  liberação, efeito que a Q1 ajuda a dimensionar.
- A base só traz as turmas de 2026/2, então não é possível dizer com que frequência cada optativa
  é oferecida.
- Não entram vagas, choques de horário ou limite de créditos, e optativas podem ter outras
  restrições que não aparecem nos requisitos.
